# 05 · Capstone — end-to-end support-policy RAG assistant

> **Core lesson:** all cells execute without external services. Read the *predict → run → explain* questions before each experiment.


## Business deliverable

Build a learning-grade assistant for the included **fictional** policy corpus. Acceptance contract:

1. Ingest documented local data with unique IDs and source provenance.
2. Create reproducible, position-aware chunks and a ranked search index.
3. Return evidence with citations; refuse when no evidence passes a threshold.
4. Report retrieval hit@k and MRR, and separate real LLM evaluation.
5. Demonstrate that the core requires **no API key or network** and no hidden model download.

**Diagram:** `JSONL -> Document -> Chunk -> TF-IDF matrix -> cosine top-k -> citation-aware evidence preview`. The Ollama path *additionally* takes this context into a true model.


In [1]:
from pathlib import Path
from RAG.src.core import (load_documents, chunk_documents, Retriever, run_query,
                          load_questions, evaluate_retrieval)
root = Path("RAG")
documents = load_documents(root / "data" / "support_policies.jsonl")
chunks = chunk_documents(documents, size=80, overlap=15)
index = Retriever(chunks)
print(f"Indexed {len(documents)} documents as {len(chunks)} chunks")


Indexed 10 documents as 10 chunks


## Query → answer + traceable citations

The output explicitly says **evidence preview** until an Ollama LLM is installed and selected. It never silently claims that retrieved text is newly generated reasoning.


In [2]:
queries = [
    "What is the free developer plan API rate limit?",
    "How many days to return an unused item?",
    "What is the CEO's birthday?",
]
for query in queries:
    result = run_query(query, index, k=3)
    print("QUESTION:", query)
    print("ANSWER:", result["answer"][:220])
    print("SOURCE IDS:", [s["doc_id"] for s in result["sources"]])
    print("---")


QUESTION: What is the free developer plan API rate limit?
ANSWER: Evidence preview (not LLM-generated): The free developer plan allows one hundred API requests per minute. The paid developer plan allows one thousand requests per minute. A request over the limit receives HTTP 429 and sh
SOURCE IDS: ['api']
---
QUESTION: How many days to return an unused item?
ANSWER: Evidence preview (not LLM-generated): After an eligible return passes warehouse inspection, the refund is sent to the original payment method within five to seven business days. Bank posting may take another two business
SOURCE IDS: ['refunds', 'returns']
---
QUESTION: What is the CEO's birthday?
ANSWER: I do not know from the provided documents.
SOURCE IDS: []
---


## Measure the index before trusting the response

A demo answer with good formatting may still come from the wrong document. Retrieval evaluation catches a different class of error from LLM hallucination testing.


In [3]:
gold = load_questions(root / "data" / "evaluation_questions.jsonl")
metrics = evaluate_retrieval(index, gold, k=3)
print("Document retrieval metrics:", metrics)
assert metrics["answerable_n"] == 10
assert metrics["unanswerable_n"] == 2


Document retrieval metrics: {'answerable_n': 10, 'unanswerable_n': 2, 'hit_at_k': 1.0, 'mrr_at_k': 0.95, 'unanswerable_abstention': 1.0}


## Run the same contract from a terminal

From the **repository root**:

```bash
conda activate awesome-rag
python -m pytest -q RAG/tests
python -m RAG.src.cli --question "How long to reset a password?"
python -m RAG.src.cli --generator ollama --question "What does a warranty exclude?"
```

The last command requires local Ollama. For a future HTTP interface, add authentication, request validation, explicit retriever/model version, timeout/quotas, structured error handling and audit logging.

## Submission / review rubric

| Dimension | Points |
|---|---:|
| Defend RAG vs LLM-only vs fine-tuning | 20 |
| Data validation, chunk lineage and test | 20 |
| Retrieval ranking plus holdout experiments | 20 |
| Citation, abstention and prompt-injection analysis | 20 |
| Conda reproducibility, tests, CI, executed outputs | 20 |

**Challenge:** Substitute your own public, license-compatible corpus; create 20 independently labeled questions; implement a dense embedding retriever and compare against the existing lexical baseline with identical data and metrics.

See [the capstone brief](../projects/support_policy_assistant/README.md).
